# Checking the Generated Constituency JSON Files

This notebook only reads the finished files. The reasoning behind the numbers and the comparison with the official margins are already covered step by step in the preparation notebook. Here the focus is on file size, structure, obvious data errors and nationwide percentage results before the files go into the app.

In [ ]:
from pathlib import Path
import math

import pandas as pd
from IPython.display import display


def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "scripts").is_dir() and (candidate / "package.json").is_file():
            return candidate
    raise RuntimeError("The notebook must run inside the repository folder.")


ROOT = find_repository_root()
GENERATED_DIRECTORY = ROOT / "scripts/data/generated/btw2021"
FIRST_VOTES_JSON = GENERATED_DIRECTORY / "first_votes.json"
SECOND_VOTES_JSON = GENERATED_DIRECTORY / "second_votes.json"

FIRST_VOTES_JSON, SECOND_VOTES_JSON

## 1. Files and sizes

In [ ]:
for path in (FIRST_VOTES_JSON, SECOND_VOTES_JSON):
    if not path.is_file():
        raise FileNotFoundError(f"File missing: {path}")

file_sizes = pd.DataFrame(
    {
        "file": [FIRST_VOTES_JSON.name, SECOND_VOTES_JSON.name],
        "MiB": [
            FIRST_VOTES_JSON.stat().st_size / 1024**2,
            SECOND_VOTES_JSON.stat().st_size / 1024**2,
        ],
    }
)

display(file_sizes)
print(f"Combined: {file_sizes['MiB'].sum():.2f} MiB")

## 2. Read the JSON and look at the first rows

In [ ]:
first_votes = pd.read_json(FIRST_VOTES_JSON)
second_votes = pd.read_json(SECOND_VOTES_JSON)

print(f"First-vote rows: {len(first_votes):,}")
print(f"Second-vote rows: {len(second_votes):,}")
display(first_votes.head(20))
display(second_votes.head(20))

## 3. Expected columns and value ranges

These checks catch malformed records before the data is copied into the application. `pandas` may read `voteType` as the integers `1` and `2`, although the JSON contract stores them as string literals. Both representations are accepted here.

In [ ]:
expected_columns = {
    "districtId",
    "state",
    "gender",
    "ageGroup",
    "party",
    "voteType",
    "electionMethod",
    "votes",
}
allowed_genders = {"m", "w"}
allowed_age_groups = {"18-24", "25-34", "35-44", "45-59", "60-69", "70+"}
allowed_methods = {"postal", "in-person"}


def check_vote_frame(frame: pd.DataFrame, label: str, expected_vote_type: int) -> None:
    missing = expected_columns - set(frame.columns)
    unexpected = set(frame.columns) - expected_columns

    print(label, {"missing": sorted(missing), "unexpected": sorted(unexpected)})
    assert not missing
    assert not unexpected

    assert frame["districtId"].notna().all()
    assert (frame["districtId"] > 0).all()
    assert frame["state"].notna().all()
    assert frame["party"].notna().all()
    assert set(frame["gender"].unique()) <= allowed_genders
    assert set(frame["ageGroup"].unique()) <= allowed_age_groups
    assert set(frame["electionMethod"].unique()) <= allowed_methods
    assert set(frame["voteType"].astype(str).unique()) == {str(expected_vote_type)}
    assert frame["votes"].map(math.isfinite).all()
    assert (frame["votes"] >= 0).all()


check_vote_frame(first_votes, "first_votes", 1)
check_vote_frame(second_votes, "second_votes", 2)

## 4. Look for duplicate detail rows

A detail row is uniquely identified by every field except `votes`. A duplicate would mean that the same demographic cell was written more than once.

In [ ]:
detail_key = [
    "districtId",
    "state",
    "gender",
    "ageGroup",
    "party",
    "voteType",
    "electionMethod",
]

duplicate_summary = pd.DataFrame(
    {
        "file": ["first_votes.json", "second_votes.json"],
        "duplicate rows": [
            int(first_votes.duplicated(detail_key).sum()),
            int(second_votes.duplicated(detail_key).sum()),
        ],
    }
)

display(duplicate_summary)
assert duplicate_summary["duplicate rows"].eq(0).all()

## 5. Look at the coverage

This overview makes missing constituencies, states, parties or demographic categories visible without assuming that every party must occur in every constituency.

In [ ]:
def coverage(frame: pd.DataFrame, label: str) -> dict[str, object]:
    return {
        "file": label,
        "rows": len(frame),
        "constituencies": frame["districtId"].nunique(),
        "states": frame["state"].nunique(),
        "parties/categories": frame["party"].nunique(),
        "genders": ", ".join(sorted(frame["gender"].unique())),
        "age groups": ", ".join(sorted(frame["ageGroup"].unique())),
        "methods": ", ".join(sorted(frame["electionMethod"].unique())),
    }


coverage_table = pd.DataFrame(
    [
        coverage(first_votes, "first_votes.json"),
        coverage(second_votes, "second_votes.json"),
    ]
)
display(coverage_table)

first_districts = set(first_votes["districtId"].unique())
second_districts = set(second_votes["districtId"].unique())

print("Only in first votes:", sorted(first_districts - second_districts))
print("Only in second votes:", sorted(second_districts - first_districts))

## 6. Example of a single constituency

The values are summed back over age, gender and election method. This is a quick way to see whether the detailed records still produce plausible constituency totals.

In [ ]:
sample_district = int(min(first_votes["districtId"].min(), second_votes["districtId"].min()))

sample_first = (
    first_votes[first_votes["districtId"] == sample_district]
    .groupby("party", as_index=False)["votes"]
    .sum()
    .sort_values("votes", ascending=False)
)
sample_second = (
    second_votes[second_votes["districtId"] == sample_district]
    .groupby("party", as_index=False)["votes"]
    .sum()
    .sort_values("votes", ascending=False)
)

print(f"Constituency {sample_district}: first votes")
display(sample_first.head(20))

print(f"Constituency {sample_district}: second votes")
display(sample_second.head(20))

## 7. Nationwide election results in percent

The following tables sum all constituencies and both election methods. Percentages use all represented party votes in the relevant group as the denominator.

The first table shows every party nationwide. For the demographic tables, the ten largest parties by nationwide second-vote share are shown separately and all remaining parties are combined as `Other parties`. This keeps the tables readable while preserving a total of 100 percent.

In [ ]:
AGE_GROUP_ORDER = ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
GENDER_ORDER = ["m", "w"]


def nationwide_party_percentages(
    first_frame: pd.DataFrame,
    second_frame: pd.DataFrame,
) -> pd.DataFrame:
    first_totals = first_frame.groupby("party")["votes"].sum()
    second_totals = second_frame.groupby("party")["votes"].sum()

    result = pd.concat(
        [
            (first_totals / first_totals.sum() * 100).rename("First vote (%)"),
            (second_totals / second_totals.sum() * 100).rename("Second vote (%)"),
        ],
        axis=1,
    ).fillna(0.0)

    return result.sort_values(
        ["Second vote (%)", "First vote (%)"],
        ascending=False,
    )


def demographic_percentage_table(
    frame: pd.DataFrame,
    group_columns: list[str],
    display_parties: list[str],
) -> pd.DataFrame:
    data = frame.loc[:, [*group_columns, "party", "votes"]].copy()
    data["displayParty"] = data["party"].where(
        data["party"].isin(display_parties),
        "Other parties",
    )

    grouped = (
        data.groupby([*group_columns, "displayParty"], as_index=False)["votes"]
        .sum()
    )
    grouped["percent"] = (
        grouped["votes"]
        / grouped.groupby(group_columns)["votes"].transform("sum")
        * 100
    )

    column_order = [*display_parties, "Other parties"]
    table = (
        grouped.pivot(
            index=group_columns,
            columns="displayParty",
            values="percent",
        )
        .fillna(0.0)
        .reindex(columns=column_order, fill_value=0.0)
    )
    table["Total"] = table.sum(axis=1)

    if "ageGroup" in group_columns:
        if len(group_columns) == 1:
            table = table.reindex(AGE_GROUP_ORDER)
        else:
            ordered_index = pd.MultiIndex.from_product(
                [GENDER_ORDER, AGE_GROUP_ORDER],
                names=["gender", "ageGroup"],
            )
            table = table.reindex(ordered_index)

    if group_columns == ["gender"]:
        table = table.reindex(GENDER_ORDER)

    return table


overall_results = nationwide_party_percentages(first_votes, second_votes)
display(overall_results.style.format("{:.2f}"))

display_parties = (
    overall_results["Second vote (%)"]
    .nlargest(10)
    .index
    .tolist()
)
print("Parties shown separately in demographic tables:", display_parties)

### First votes by age group and gender

In [ ]:
print("First votes by age group")
display(
    demographic_percentage_table(
        first_votes,
        ["ageGroup"],
        display_parties,
    ).style.format("{:.2f}")
)

print("First votes by gender")
display(
    demographic_percentage_table(
        first_votes,
        ["gender"],
        display_parties,
    ).style.format("{:.2f}")
)

print("First votes by gender and age group")
display(
    demographic_percentage_table(
        first_votes,
        ["gender", "ageGroup"],
        display_parties,
    ).style.format("{:.2f}")
)

### Second votes by age group and gender

In [ ]:
print("Second votes by age group")
display(
    demographic_percentage_table(
        second_votes,
        ["ageGroup"],
        display_parties,
    ).style.format("{:.2f}")
)

print("Second votes by gender")
display(
    demographic_percentage_table(
        second_votes,
        ["gender"],
        display_parties,
    ).style.format("{:.2f}")
)

print("Second votes by gender and age group")
display(
    demographic_percentage_table(
        second_votes,
        ["gender", "ageGroup"],
        display_parties,
    ).style.format("{:.2f}")
)